# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam271/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
# ML-06 setup — load dataset while auditing malformed rows

import pandas as pd
import numpy as np
import os

DATA_PATH = "content_refresh_anonymized.csv"

print("File exists:", os.path.exists(DATA_PATH))
print("File path:", DATA_PATH)

# Track malformed rows instead of silently hiding them.
bad_rows = []

def handle_bad_line(bad_line):
    bad_rows.append(bad_line)
    return None

df = pd.read_csv(
    DATA_PATH,
    engine="python",
    on_bad_lines=handle_bad_line
)

print("\nRows loaded:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())
print("Malformed rows skipped:", len(bad_rows))

print("\nData loaded successfully with malformed rows explicitly tracked.")

assert len(df.columns) == 44
assert "content_id" in df.columns
assert "client_id" in df.columns

File exists: True
File path: content_refresh_anonymized.csv

Rows loaded: 45975
Columns: 44
Clients: 34
Malformed rows skipped: 3

Data loaded successfully with malformed rows explicitly tracked.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Distributions

Before testing the signals, I first inspect the distributions of the main traffic, search, content, and freshness fields. Traffic variables can be highly skewed, so I use medians and percentiles to understand their typical values and heavy tails.

I also check `avg_position` separately because a value of 0 represents missing position data rather than an actual search position.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Distribution audit for the main signal fields

distribution_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "search_volume",
    "word_count",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

distribution_summary = df[distribution_cols].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
).T

distribution_summary = distribution_summary[
    ["count", "mean", "50%", "75%", "90%", "99%", "max"]
].round(2)

print(distribution_summary)

print("\nTraffic fields — raw vs log1p median:")
for col in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    print(
        f"{col}: "
        f"raw median={df[col].median():.2f}, "
        f"log1p median={np.log1p(df[col]).median():.2f}"
    )

print("\nMissing position values:")
print("avg_position == 0:", (df["avg_position"] == 0).sum())

print("\nDistribution audit completed.")

                          count     mean     50%     75%      90%       99%  \
impressions_90d         45973.0  5210.75   726.0  3585.0  12070.0  72958.44   
clicks_90d              45973.0    16.13     1.0     7.0     32.0    255.00   
sessions_90d            45973.0    37.00     7.0    27.0     88.0    452.00   
word_count              34195.0  3107.49  2875.0  3662.0   5333.0   7299.36   
days_since_last_update  45973.0    46.17    20.0   104.0    104.0    106.00   

                             max  
impressions_90d         517715.0  
clicks_90d                4178.0  
sessions_90d              4345.0  
word_count                9546.0  
days_since_last_update     373.0  

Traffic fields — raw vs log1p median:
impressions_90d: raw median=726.00, log1p median=6.59
clicks_90d: raw median=1.00, log1p median=0.69
sessions_90d: raw median=7.00, log1p median=2.08

Missing position values:
avg_position == 0: 0

Distribution audit completed.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

I test three directional signals using grouped medians rather than relying only on raw averages. This reduces the influence of the strong right-skew visible in the distributions.

### Signal #1 — Search visibility and clicks
Test whether pages with more impressions generally have more clicks.

### Signal #2 — CTR and clicks
Test whether pages with higher CTR generally have more clicks.

### Signal #3 — Content freshness and impressions
Test whether more recently updated content generally has stronger search visibility.

Each signal receives a data-driven verdict: `CONFIRMED`, `MIXED`, or `OPPOSITE`. These are observed associations, not causal claims.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Test three directional signals using quartile groups and median outcomes

# ML-06 — Three signal tests
# Convert numeric signal fields safely before analysis.

# Keep the original dataframe unchanged except for analysis helper columns.
df["ctr_numeric"] = pd.to_numeric(df["ctr"], errors="coerce")


# -----------------------------
# Signal #1: Impressions -> Clicks
# -----------------------------

df["impression_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)

signal_1 = (
    df.groupby("impression_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median")
      )
      .reset_index()
)

print("SIGNAL #1 — Impressions vs clicks")
print(signal_1.round(2))

s1_changes = signal_1["median_clicks"].diff().dropna()

if (s1_changes >= 0).all() and (s1_changes > 0).any():
    s1_verdict = "CONFIRMED"
elif (s1_changes <= 0).all() and (s1_changes < 0).any():
    s1_verdict = "OPPOSITE"
else:
    s1_verdict = "MIXED"

print("\nVerdict:", s1_verdict)


# -----------------------------
# Signal #2: CTR -> Clicks
# -----------------------------

ctr_data = df.dropna(subset=["ctr_numeric"]).copy()

ctr_data["ctr_bucket"] = pd.qcut(
    ctr_data["ctr_numeric"],
    q=4,
    duplicates="drop"
)

signal_2 = (
    ctr_data.groupby("ctr_bucket", observed=True)
            .agg(
                n=("content_id", "size"),
                median_ctr=("ctr_numeric", "median"),
                median_clicks=("clicks_90d", "median")
            )
            .reset_index()
)

print("\nSIGNAL #2 — CTR vs clicks")
print(signal_2.round(4))

s2_changes = signal_2["median_clicks"].diff().dropna()

if (s2_changes >= 0).all() and (s2_changes > 0).any():
    s2_verdict = "CONFIRMED"
elif (s2_changes <= 0).all() and (s2_changes < 0).any():
    s2_verdict = "OPPOSITE"
else:
    s2_verdict = "MIXED"

print("\nVerdict:", s2_verdict)


# -----------------------------
# Signal #3: Freshness -> Impressions
# -----------------------------

fresh_data = df.dropna(subset=["days_since_last_update"]).copy()

fresh_data["freshness_bucket"] = pd.qcut(
    fresh_data["days_since_last_update"],
    q=4,
    duplicates="drop"
)

signal_3 = (
    fresh_data.groupby("freshness_bucket", observed=True)
              .agg(
                  n=("content_id", "size"),
                  median_days_since_update=("days_since_last_update", "median"),
                  median_impressions=("impressions_90d", "median")
              )
              .reset_index()
)

print("\nSIGNAL #3 — Freshness vs impressions")
print(signal_3.round(2))

# Lower days_since_last_update = more recently updated.
# Therefore decreasing impressions as age increases supports the signal.

s3_changes = signal_3["median_impressions"].diff().dropna()

if (s3_changes <= 0).all() and (s3_changes < 0).any():
    s3_verdict = "CONFIRMED"
elif (s3_changes >= 0).all() and (s3_changes > 0).any():
    s3_verdict = "OPPOSITE"
else:
    s3_verdict = "MIXED"

print("\nVerdict:", s3_verdict)


# -----------------------------
# Data-quality / sample-size checks
# -----------------------------

print("\nData-quality checks:")
print("CTR values converted to numeric:", ctr_data.shape[0])
print("CTR values unable to convert:", df["ctr_numeric"].isna().sum())

print("\nMinimum group sizes:")
print("Signal #1:", signal_1["n"].min())
print("Signal #2:", signal_2["n"].min())
print("Signal #3:", signal_3["n"].min())

SIGNAL #1 — Impressions vs clicks
    impression_bucket      n  median_impressions  median_clicks
0       (0.999, 80.0]  11497                 9.0            0.0
1       (80.0, 726.0]  11493               296.0            0.0
2     (726.0, 3585.0]  11492              1613.5            2.0
3  (3585.0, 517715.0]  11491              9564.0           22.0

Verdict: CONFIRMED

SIGNAL #2 — CTR vs clicks
       ctr_bucket      n  median_ctr  median_clicks
0  (-0.001, 0.07]  23384        0.00            0.0
1    (0.07, 0.28]  11136        0.16            5.0
2   (0.28, 100.0]  11452        0.54           10.0

Verdict: CONFIRMED

SIGNAL #3 — Freshness vs impressions
  freshness_bucket      n  median_days_since_update  median_impressions
0    (0.999, 20.0]  24306                      20.0               361.0
1    (20.0, 104.0]  21174                     104.0              1253.5
2   (104.0, 373.0]    493                     183.0                26.0

Verdict: MIXED

Data-quality checks:
CTR val

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The baseline score uses search visibility and CTR as signals for content-review prioritization. I test the CTR part of this assumption among pages with meaningful search visibility.

The test compares CTR groups using median clicks while keeping the group sizes visible.

This evaluates whether lower CTR is associated with weaker click performance in the observed data. It does not prove that low CTR is caused by a content problem.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Flag-linked test:
# Among pages with meaningful search visibility,
# test whether higher CTR is associated with more clicks.

flag_data = df[
    (df["impressions_90d"] >= 100) &
    (df["ctr_numeric"].notna())
].copy()

flag_data["ctr_group"] = pd.qcut(
    flag_data["ctr_numeric"],
    q=4,
    duplicates="drop"
)

flag_test = (
    flag_data.groupby("ctr_group", observed=True)
             .agg(
                 n=("content_id", "size"),
                 median_ctr=("ctr_numeric", "median"),
                 median_impressions=("impressions_90d", "median"),
                 median_clicks=("clicks_90d", "median")
             )
             .reset_index()
)

print("FLAG-LINKED TEST — Pages with >=100 impressions")
print(flag_test.round(4))

flag_changes = flag_test["median_clicks"].diff().dropna()

if (flag_changes >= 0).all() and (flag_changes > 0).any():
    flag_verdict = "CONFIRMED"
elif (flag_changes <= 0).all() and (flag_changes < 0).any():
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

print("\nVerdict:", flag_verdict)

print("\nMinimum group size:", flag_test["n"].min())

assert len(flag_data) >= 50
assert flag_test["n"].min() >= 50

FLAG-LINKED TEST — Pages with >=100 impressions
        ctr_group      n  median_ctr  median_impressions  median_clicks
0  (-0.001, 0.14]  16870        0.00              1039.0            0.0
1    (0.14, 0.34]   8522        0.23              2900.0            6.0
2   (0.34, 11.76]   8299        0.57              2600.0           15.0

Verdict: CONFIRMED

Minimum group size: 8299


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit provides support for using impressions and CTR as directional signals for content-review prioritization. Higher-impression groups show higher median clicks, and higher-CTR groups also show higher median clicks.

The freshness signal is mixed: the observed relationship between update age and impressions is not consistently ordered across the groups. Therefore, freshness should not be treated as a standalone indicator of search visibility.

The flag-linked test is confirmed in the observed data: among pages with at least 100 impressions, median clicks increase from 0 in the lowest-CTR group to 6 in the middle group and 15 in the highest-CTR group.

For the content team, these signals can help prioritize pages for review, but they are decision-support signals rather than proof of causation. Page context, search intent, and other content factors should still be considered before making a refresh decision.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final ML-06 audit summary

print("ML-06 — Signal Audit completed")
print("--------------------------------")
print("Rows loaded:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())
print("Malformed rows skipped:", len(bad_rows))
print()
print("Signal #1 — Impressions vs clicks:", s1_verdict)
print("Signal #2 — CTR vs clicks:", s2_verdict)
print("Signal #3 — Freshness vs impressions:", s3_verdict)
print("Flag-linked test:", flag_verdict)
print()
print("Practical conclusion:")
print("- Impressions and CTR provide useful directional signals.")
print("- Freshness shows a mixed relationship with impressions.")
print("- The flag-linked CTR assumption is supported in the observed data.")
print("- These findings are associations, not causal claims.")


ML-06 — Signal Audit completed
--------------------------------
Rows loaded: 45975
Columns: 46
Clients: 34
Malformed rows skipped: 3

Signal #1 — Impressions vs clicks: CONFIRMED
Signal #2 — CTR vs clicks: CONFIRMED
Signal #3 — Freshness vs impressions: MIXED
Flag-linked test: CONFIRMED

Practical conclusion:
- Impressions and CTR provide useful directional signals.
- Freshness shows a mixed relationship with impressions.
- The flag-linked CTR assumption is supported in the observed data.
- These findings are associations, not causal claims.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.